In [3]:
import os
import pathlib
import matplotlib.pyplot as plt
import json
from model_functions import test
import torch
from medmnist import ChestMNIST
from torchvision.transforms import v2
from model import ResNet18

node_logs = []

nodes = 10

tf = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32,scale=True)
])

test_data = ChestMNIST(split='test',transform=tf,download=True,root="./data")
val_data = ChestMNIST(split='val',transform=tf,download=True,root="./data")

classes = len(test_data.info["label"])
channels = test_data.info["n_channels"]
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ResNet18(channels,classes).to(device)


In [4]:
local_test_results = []

test_dl = torch.utils.data.DataLoader(test_data,16384)

for i in range(nodes):
    node = pathlib.Path(f"./results/node_{i}").resolve()
    model.load_state_dict(torch.load(node/"best_model.pth"))
    if node.exists():
        print(f"Node {i}")
        best_model = node.glob("*.pth")
        _,test_acc = test(test_dl,model,loss_fn=torch.nn.BCEWithLogitsLoss(),device=device)
        local_test_results.append((f"node_{i}",test_acc))


Node 0
Test Error: Accuracy: 94.7%, Average Loss: 0.189428
Node 1
Test Error: Accuracy: 94.7%, Average Loss: 0.179769
Node 2
Test Error: Accuracy: 94.7%, Average Loss: 0.179776
Node 3
Test Error: Accuracy: 94.7%, Average Loss: 0.195576
Node 4
Test Error: Accuracy: 94.7%, Average Loss: 0.182728
Node 5
Test Error: Accuracy: 94.7%, Average Loss: 0.196224
Node 6
Test Error: Accuracy: 94.7%, Average Loss: 0.179276
Node 7
Test Error: Accuracy: 94.7%, Average Loss: 0.194751
Node 8
Test Error: Accuracy: 94.7%, Average Loss: 0.177780
Node 9
Test Error: Accuracy: 94.7%, Average Loss: 0.185952


In [5]:
global_models_results = []

global_model_dir = pathlib.Path(f"./results/global_model").resolve()

best_val_global_model = None
best_val_acc = -1

val_dl = torch.utils.data.DataLoader(val_data,16384)
test_dl = torch.utils.data.DataLoader(test_data,16384)

models = global_model_dir.glob("*.pth")
for gm in models:
    print(gm)
    model.load_state_dict(torch.load(gm))
    _,val_acc = test(test_dl,model,loss_fn=torch.nn.BCEWithLogitsLoss(),device=device)
    if val_acc > best_val_acc:
        best_val_global_model = gm
        best_val_acc = val_acc
    
if best_val_global_model != None:
    model.load_state_dict(torch.load(best_val_global_model))
    _,test_acc = test(test_dl,model,loss_fn=torch.nn.BCEWithLogitsLoss(),device=device)
    global_models_results.append((best_val_global_model.name,test_acc))

/mnt/sdc/Hasan/Documents/My Documents/Code/GitLab/kademlia-based-federated-learning/results/global_model/global_model_1778487205.0304012.pth
Test Error: Accuracy: 94.7%, Average Loss: 0.198870
/mnt/sdc/Hasan/Documents/My Documents/Code/GitLab/kademlia-based-federated-learning/results/global_model/global_model_1778488641.9564013.pth
Test Error: Accuracy: 94.7%, Average Loss: 0.180568
/mnt/sdc/Hasan/Documents/My Documents/Code/GitLab/kademlia-based-federated-learning/results/global_model/global_model_1778490080.7365277.pth
Test Error: Accuracy: 94.7%, Average Loss: 0.177724
/mnt/sdc/Hasan/Documents/My Documents/Code/GitLab/kademlia-based-federated-learning/results/global_model/global_model_1778491541.168527.pth
Test Error: Accuracy: 94.7%, Average Loss: 0.177787
/mnt/sdc/Hasan/Documents/My Documents/Code/GitLab/kademlia-based-federated-learning/results/global_model/global_model_1778492981.1250522.pth
Test Error: Accuracy: 94.6%, Average Loss: 0.178957
/mnt/sdc/Hasan/Documents/My Document

In [6]:
print(local_test_results)
print(global_models_results)

[('node_0', 0.9474339461634491), ('node_1', 0.947239716998507), ('node_2', 0.9469435971240872), ('node_3', 0.9474148416554222), ('node_4', 0.947029567410209), ('node_5', 0.9474371302481203), ('node_6', 0.9472142443211371), ('node_7', 0.9473766326393681), ('node_8', 0.9471155376963305), ('node_9', 0.9473384236233136)]
[('global_model_1778487205.0304012.pth', 0.9474180257400932)]


In [7]:
import pandas as pd

pd.DataFrame(local_test_results)

,0,1
0,node_0,0.947434
1,node_1,0.947240
2,node_2,0.946944
3,node_3,0.947415
4,node_4,0.947030
5,node_5,0.947437
6,node_6,0.947214
7,node_7,0.947377
8,node_8,0.947116
9,node_9,0.947338


In [8]:
from numpy import average

node_results = [node[1] for node in local_test_results]
best_node = max(node_results)
average_node = average(node_results)
worst_node = min(node_results)

print(f"Best Node Accuracy : {best_node}")
print(f"Average Node Accuracy : {average_node}")
print(f"Worst Node Accuracy : {worst_node}")

Best Node Accuracy : 0.9474371302481203
Average Node Accuracy : 0.9472543637879944
Worst Node Accuracy : 0.9469435971240872


In [9]:
import pandas as pd

# the test result from the best global model from validation accuracy testing

pd.DataFrame(global_models_results)

,0,1
0,global_model_1778487205.0304012.pth,0.947418
